In [1]:
import pandas as pd
from fluwatch.warehouse.load import *

In [2]:
df_week = load_flu_week()
dim_state = load_dim_state()
dim_time = load_dim_time()

In [3]:
tables = {
    "fact_flu_week (df_week)": df_week,
    "dim_state": dim_state,
    "dim_time": dim_time,
}

for name, df in tables.items():
    print("=" * 80)
    print(f"BẢNG: {name} | Kích thước: {df.shape[0]:,} dòng × {df.shape[1]} cột")
    print("-" * 80)
    
    schema_df = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "example_value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
    })
    print(schema_df.to_string())
    
    print("\n--- 5 dòng dữ liệu mẫu ---")
    print(df.head(5).to_string())
    print("\n")

BẢNG: fact_flu_week (df_week) | Kích thước: 44,931 dòng × 90 cột
--------------------------------------------------------------------------------
                             dtype  non_null  null_pct        example_value
state                       string     44931      0.00                   ak
week                datetime64[us]     44931      0.00  2010-01-02 00:00:00
epiweek                      Int64     44931      0.00               200952
season                      string     44931      0.00              2009-10
season_week                  Int64     44931      0.00                   13
ili_pct                    float64     41540      7.55             0.875146
num_ili                      Int64     41580      7.46                   15
num_patients                 Int64     41580      7.46                 1714
hosp_flu                     Int64     16224     63.89                    0
rate_flu                   float64     16224     63.89                  0.0
cdc_level         

In [4]:
season_cov = df_week.groupby("season").agg(
    n_weeks=("epiweek", "nunique"),
    min_week=("epiweek", "min"),
    max_week=("epiweek", "max"),
    ili_missing_pct=("ili_pct", lambda s: round(s.isna().mean() * 100, 1)),
    nssp_missing_pct=("ed_pct_flu", lambda s: round(s.isna().mean() * 100, 1)) if "ed_pct_flu" in df.columns else ("ili_pct", lambda s: np.nan)
)
print("=== ĐỘ PHỦ THEO MÙA ===")
print(season_cov.to_string())

# 2. Liệt kê các cặp (bang, mùa) có tỷ lệ thiếu ili_pct > 30%
state_season_miss = (
    df_week.groupby(["state", "season"])["ili_pct"]
    .apply(lambda s: round(s.isna().mean() * 100, 1))
    .reset_index(name="ili_miss_pct")
)
high_miss = state_season_miss[state_season_miss["ili_miss_pct"] > 30]
print("\n=== CÁC CẶP BANG × MÙA THIẾU > 30% ILINET ===")
print(high_miss.pivot(index="state", columns="season", values="ili_miss_pct").fillna("-").to_string())

# 3. Kiểm tra riêng 8 bang thiếu ILINet mùa 2025-26 có đủ NSSP (ed_pct_flu) không
if "ed_pct_flu" in df.columns:
    s2526 = df[df["season"] == "2025-26"].groupby("state").agg(
        ili_miss=("ili_pct", lambda s: round(s.isna().mean() * 100, 1)),
        nssp_miss=("ed_pct_flu", lambda s: round(s.isna().mean() * 100, 1)),
    )
    print("\n=== TÌNH HÌNH BÙ NSSP MÙA 2025-26 ===")
    print(s2526[s2526["ili_miss"] > 30].to_string())

=== ĐỘ PHỦ THEO MÙA ===
         n_weeks  min_week  max_week  ili_missing_pct  nssp_missing_pct
season                                                                 
2009-10       40    200952    201039            100.0               NaN
2010-11       52    201040    201139              4.6               NaN
2011-12       52    201140    201239              3.9               NaN
2012-13       52    201240    201339              4.0               NaN
2013-14       52    201340    201439              4.0               NaN
2014-15       53    201440    201539              4.3               NaN
2015-16       52    201540    201639              4.0               NaN
2016-17       52    201640    201739              2.0               NaN
2017-18       52    201740    201839              2.1               NaN
2018-19       52    201840    201939              2.0               NaN
2019-20       52    201940    202039              2.0               NaN
2020-21       53    202040    202139    

C:\Users\duysa\AppData\Local\Temp\ipykernel_12876\2759273518.py:19: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  print(high_miss.pivot(index="state", columns="season", values="ili_miss_pct").fillna("-").to_string())


In [5]:
states_8 = ["ak", "ct", "hi", "ny", "ok", "or", "sd", "ut"]

check_nssp = (
    df_week[df_week["season"].isin(["2025-26", "2026-27"]) & df_week["state"].isin(states_8)]
    .groupby(["season", "state"])
    .agg(
        ili_miss_pct=("ili_pct", lambda s: round(s.isna().mean() * 100, 1)),
        nssp_miss_pct=("ed_pct_flu", lambda s: round(s.isna().mean() * 100, 1)),
    )
)
print(check_nssp.to_string())

               ili_miss_pct  nssp_miss_pct
season  state                             
2025-26 ak            100.0            0.0
        ct            100.0            0.0
        hi            100.0            0.0
        ny            100.0            0.0
        ok            100.0            0.0
        or            100.0            0.0
        sd            100.0            0.0
        ut            100.0            0.0
2026-27 ak              0.0            0.0
        ct              0.0            0.0
        hi              0.0            0.0
        ny              0.0            0.0
        ok              0.0            0.0
        or              0.0            0.0
        sd              0.0            0.0
        ut              0.0            0.0
